# Lesson 2: The SwiGLU Feed-Forward Network

In this notebook, we explore the **SwiGLU (Swish Gated Linear Unit)** architecture used in **Llama 3, DeepSeek, and Mirendil-Base**.

---

## 1. Why Gating Matters: The Intuition
In a classic MLP:
$$\text{FFN}(x) = \text{Activation}(x W_1) W_2$$

In SwiGLU, there are **two parallel projection paths**:
1. **Up-projection (`w_up`):** Creates the candidate feature values.
2. **Gate-projection (`w_gate`):** Learns a continuous mask (using $\text{SiLU}$) that controls how much of each feature passes through.

$$\text{SwiGLU}(x) = \Big( \text{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}}) \Big) W_{\text{down}}$$

The element-wise product $(\odot)$ acts like a **learned volume dial** for every neuron.

In [ ]:
import sys
from pathlib import Path
# Ensure project root is in sys.path
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

import torch
import torch.nn as nn
import torch.nn.functional as F
from src.model.mlp import SwiGLUFFN

print(f"PyTorch version: {torch.__version__}")

## 2. Comparing the Activations: ReLU vs GELU vs SiLU (Swish)

Let's see why $\text{SiLU}(z) = z \cdot \text{sigmoid}(z)$ is preferred over older functions.

In [ ]:
# Input range from -4 to +4
z = torch.linspace(-4, 4, steps=9)

relu_out = F.relu(z)
gelu_out = F.gelu(z)
silu_out = F.silu(z)

print(f"{'Input (z)':<10} | {'ReLU':<10} | {'GELU':<10} | {'SiLU (Swish)':<12}")
print("-" * 50)
for i in range(len(z)):
    print(f"{z[i].item():<10.2f} | {relu_out[i].item():<10.2f} | {gelu_out[i].item():<10.2f} | {silu_out[i].item():<12.2f}")

### Key Observation:
- **ReLU** hard-clips everything negative to `0.0`. When gradients hit 0, neurons "die" and stop learning.
- **SiLU (Swish)** is smooth, non-monotonic, and allows a small negative gradient to pass through (e.g. at $z = -1.0$, $\text{SiLU}(z) \approx -0.27$). This keeps gradient flow healthy during deep pretraining.

## 3. The 8/3 Hidden Dimension Rule
Let's see how `SwiGLUFFN` automatically computes its `hidden_dim` from model dimension `dim`.

In [ ]:
dim = 768  # Standard base model dimension

# 8/3 formula: 2 * (4 * dim) / 3
raw_hidden_dim = int(2 * (4 * dim) / 3)
print(f"Model dim: {dim}")
print(f"Raw 8/3 hidden dim: {raw_hidden_dim}")

# Round up to nearest multiple of 256 for optimal GPU tensor core alignment
multiple_of = 256
aligned_hidden_dim = multiple_of * ((raw_hidden_dim + multiple_of - 1) // multiple_of)
print(f"Aligned hidden dim (multiple of 256): {aligned_hidden_dim}")

## 4. Live Forward Pass: Inspecting Shapes and Gating
Let's instantiate our `SwiGLUFFN` and pass dummy tokens through it.

In [ ]:
# Batch size = 2, Sequence length = 4 tokens, Model dim = 768
ffn = SwiGLUFFN(dim=dim, hidden_dim=aligned_hidden_dim)
x = torch.randn(2, 4, dim)

print(f"Input shape: {x.shape}")

# Let's trace each internal tensor:
gate = ffn.w_gate(x)               # [2, 4, 2048]
activated_gate = F.silu(gate)      # [2, 4, 2048]
up = ffn.w_up(x)                   # [2, 4, 2048]
gated_features = activated_gate * up  # [2, 4, 2048]
output = ffn.w_down(gated_features)   # [2, 4, 768]

print(f"Gate projection shape:     {gate.shape}")
print(f"Up projection shape:       {up.shape}")
print(f"Gated (multiplied) shape:  {gated_features.shape}")
print(f"Final Down-project shape:  {output.shape}")

# Verify full forward pass matches manual trace
full_out = ffn(x)
assert torch.allclose(output, full_out, atol=1e-6)
print("\nSUCCESS: SwiGLU forward pass fully verified!")